_This Notebook calculates the SPEI-records for the different KNMI-stations_

30 years of data is downloaded as this is good "common" practice <br>

https://iahs.info/uploads/dms/16617.66-367-373-363-55-Paper-213-Stagge.pdf

In [ ]:
de_bilt_path = r"D:\Thesis\10.Thesis_Data\00_environmental_variables_raw\de_Bilt.txt"
deelen_path = r"D:\Thesis\10.Thesis_Data\00_environmental_variables_raw\Deelen.txt"
heino_path = r"D:\Thesis\10.Thesis_Data\00_environmental_variables_raw\Heino.txt"
lelystad_path = r"D:\Thesis\10.Thesis_Data\00_environmental_variables_raw\Lelystad.txt"

In [ ]:
# READ KNMI DATA
import pandas as pd
import io
import warnings

def read_knmi_data(filepath):
    """
    Read KNMI daily climate data file and return a cleaned DataFrame.
    
    Parameters:
    -----------
    filepath : str
        Path to the KNMI data file
    
    Returns:
    --------
    pd.DataFrame
        DataFrame with parsed date and scaled values (°C, mm)
    """
    
    # Read the file and find where data starts
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Find the line where actual data starts (skip the header line with column names)
    data_start = 0
    for i, line in enumerate(lines):
        if line.strip().startswith('STN,YYYYMMDD'):
            data_start = i + 1
            break
    
    # Filter out any remaining comment or header lines
    data_lines = []
    for line in lines[data_start:]:
        # Skip empty lines and lines starting with #
        stripped = line.strip()
        if stripped and not stripped.startswith('#'):
            data_lines.append(line)
    
    # Read the data portion into a DataFrame
    data_text = ''.join(data_lines)
    df = pd.read_csv(
        io.StringIO(data_text), 
        names=['STN', 'YYYYMMDD', 'TG', 'RH', 'EV24'],
        skipinitialspace=True,
        dtype={'STN': int, 'YYYYMMDD': str, 'TG': float, 'RH': float, 'EV24': float}
    )
    
    # Parse the date column
    df['date'] = pd.to_datetime(df['YYYYMMDD'], format='%Y%m%d')
    
    df['TG'] = df['TG'] / 10
    df['RH'] = df['RH'] / 10
    df['EV24'] = df['EV24'] / 10
    
    # Handle special precipitation values (-1 means < 0.05 mm)
    df.loc[df['RH'] == -0.1, 'RH'] = 0.05
    
    # Select and rename columns
    df = df[['STN', 'date', 'TG', 'RH', 'EV24']]
    df.columns = ['station', 'date', 'temp_c', 'precip_mm', 'pet_mm']
    
    # Set date as index
    df = df.set_index('date')
    
    # Check if DataFrame is empty and issue warning
    if df.empty:
        warnings.warn(f"Warning: No data was found in the file {filepath}."
                     f"The returned DataFrame is empty. Please check the file format and content.",
                     UserWarning,
                     stacklevel=2)
    
    return df

In [ ]:
no_to_station_dict = {
    260: 'de Bilt',
    269: 'Lelystad',
    275: 'Deelen',
    278: 'Heino',
}

In [ ]:
de_bilt_df = read_knmi_data(de_bilt_path)
deelen_df = read_knmi_data(deelen_path)
heino_df = read_knmi_data(heino_path)
lelystad_df = read_knmi_data(lelystad_path)

In [ ]:
# Aligning the dfs
def align_dataframes_complete(dfs, df_names=None):
    """
    Align to common period where ALL dfs have complete data
    """
    if df_names is None:
        df_names = [f"df_{i}" for i in range(len(dfs))]
    
    # Find first complete date for each
    first_complete_dates = []
    for name, df in zip(df_names, dfs):
        complete_rows = df.dropna(how='any')
        if len(complete_rows) > 0:
            first_complete_dates.append(complete_rows.index[0])
            print(f"{name} first complete: {complete_rows.index[0]}")
    
    # Find last complete date for each
    last_complete_dates = []
    for name, df in zip(df_names, dfs):
        complete_rows = df.dropna(how='any')
        if len(complete_rows) > 0:
            last_complete_dates.append(complete_rows.index[-1])
            print(f"{name} last complete: {complete_rows.index[-1]}")
    
    # Common period
    common_start = max(first_complete_dates)
    common_end = min(last_complete_dates)
    
    print(f"\nCommon complete period: {common_start} to {common_end}")
    
    # Trim all
    trimmed_dfs = [df.loc[common_start:common_end].copy() for df in dfs]
    
    return trimmed_dfs

In [ ]:
# Usage
aligned_dfs = align_dataframes_complete(
    [de_bilt_df, deelen_df, heino_df, lelystad_df],
    ['De Bilt', 'Deelen', 'Heino', 'Lelystad']
)
de_bilt_aligned_df, deelen_aligned_df, heino_aligned_df, lelystad_aligned_df = aligned_dfs

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from lmoments3 import distr


def _fit_glo(values):
    """
    Fit 3-parameter generalized logistic distribution via L-moments.
    
    The generalized logistic in lmoments3 (Hosking's parameterization) is the
    distribution used by the SPEI package in R for the log-logistic fit in
    Vicente-Serrano et al. (2010).

    Returns the parameter dict {'k', 'loc', 'scale'} or None on failure.
    """
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) < 30:
        return None
    try:
        return distr.glo.lmom_fit(values)
    except ValueError as e:
        # lmom_fit raises ValueError for degenerate samples
        # (e.g. "L-Moments invalid" when |t3| >= 1). Let everything else surface.
        print(f"    [warning] glo fit failed: {e}")
        return None


def _cdf_to_spei(cdf_values):
    """Inverse standard normal of the fitted CDF, clipped to avoid +/- inf."""
    cdf_values = np.clip(cdf_values, 1e-6, 1 - 1e-6)
    return stats.norm.ppf(cdf_values)


def aggregate_to_monthly(df, precip_col="precip_mm", pet_col="pet_mm",
                         min_coverage=0.9):
    """
    Aggregate daily P and PET to monthly totals.

    Months with less than `min_coverage` of days present are set to NaN to
    avoid biasing the water balance with partial months.
    """
    daily = df[[precip_col, pet_col]].copy()

    monthly_sum = daily.resample("MS").sum(min_count=1)
    days_present = daily.resample("MS").count()
    days_in_month = daily.resample("MS").size().to_frame("n").assign(
        days_in_month=lambda x: x.index.days_in_month
    )["days_in_month"]

    coverage = days_present.div(days_in_month, axis=0)
    mask = (coverage >= min_coverage).all(axis=1)
    monthly_sum.loc[~mask, :] = np.nan

    monthly_sum["water_balance"] = monthly_sum[precip_col] - monthly_sum[pet_col]
    return monthly_sum


def calculate_spei_monthly(monthly_df, timescales=range(1, 13),
                           wb_col="water_balance"):
    """
    Calculate SPEI-1 through SPEI-n at monthly resolution.

    For each timescale k, the water balance is accumulated over k months,
    then a 3-parameter log-logistic is fitted *separately for each calendar
    month* (12 fits per timescale) and the CDF is transformed to a standard
    normal variate.

    Parameters
    ----------
    monthly_df : DataFrame with monthly DatetimeIndex (month start) and a
        water balance column.
    timescales : iterable of integer month accumulation lengths.
    wb_col : water balance column name.

    Returns
    -------
    DataFrame with columns 'SPEI_1', 'SPEI_2', ..., aligned to monthly_df.index.
    """
    results = {}
    diagnostics = {}

    for k in timescales:
        accumulated = monthly_df[wb_col].rolling(
            window=k, min_periods=k
        ).sum()

        spei_k = pd.Series(np.nan, index=monthly_df.index)
        fit_info = {}

        for month in range(1, 13):
            month_mask = accumulated.index.month == month
            month_values = accumulated[month_mask].dropna()

            if len(month_values) < 30:
                fit_info[month] = {"n": len(month_values), "fitted": False}
                continue

            params = _fit_glo(month_values.values)
            if params is None:
                fit_info[month] = {"n": len(month_values), "fitted": False}
                continue

            cdf_vals = distr.glo.cdf(month_values.values, **params)
            spei_k.loc[month_values.index] = _cdf_to_spei(cdf_vals)
            fit_info[month] = {"n": len(month_values), "fitted": True,
                               "params": params}

        results[f"SPEI_{k}"] = spei_k
        diagnostics[k] = fit_info

    return pd.DataFrame(results), diagnostics

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.dates import YearLocator, MonthLocator, DateFormatter


def run_sanity_checks(spei_df, station_name):
    """Print sanity checks for fitted SPEI values."""
    print(f"\n{'='*70}")
    print(f"SPEI sanity checks — {station_name}")
    print(f"{'='*70}")

    # 1. Distribution: each timescale should be ~standard normal.
    print("\n[1] Overall distribution (expect mean~0, std~1):")
    desc = spei_df.describe().loc[["mean", "std", "min", "max", "count"]]
    print(desc.round(2).to_string())

    # 2. Seasonal bias: mean SPEI should be ~0 in every calendar month.
    print("\n[2] Seasonal bias check (max |monthly mean| per timescale):")
    print("    Values > 0.1 suggest the per-month fit is leaking seasonality.")
    for col in spei_df.columns:
        seasonal_mean = spei_df[col].groupby(spei_df.index.month).mean()
        max_bias = seasonal_mean.abs().max()
        flag = "  <-- check" if max_bias > 0.1 else ""
        print(f"    {col:<10} max |mean| = {max_bias:.3f}{flag}")

    # 3. McKee drought event counts.
    print("\n[3] Month counts by McKee category:")
    print(f"    {'timescale':<12}{'extreme dry':>14}{'severe dry':>14}"
          f"{'moderate dry':>14}{'moderate wet':>14}{'severe wet':>14}")
    for col in spei_df.columns:
        s = spei_df[col].dropna()
        ext_d = (s < -2.0).sum()
        sev_d = ((s >= -2.0) & (s < -1.5)).sum()
        mod_d = ((s >= -1.5) & (s < -1.0)).sum()
        mod_w = ((s > 1.0) & (s <= 1.5)).sum()
        sev_w = (s > 1.5).sum()
        print(f"    {col:<12}{ext_d:>14}{sev_d:>14}{mod_d:>14}"
              f"{mod_w:>14}{sev_w:>14}")
    print()


def plot_spei_monthly(df,
                     plot_timescales=(1, 3, 6, 12),
                     all_timescales=range(1, 13),
                     station_dict=None,
                     start_date=None,
                     end_date=None,
                     figsize=(16, 10),
                     run_checks=True):
    """
    Plot monthly SPEI as signed bars for selected timescales.

    Parameters
    ----------
    df : DataFrame with daily DatetimeIndex and columns 'station',
        'precip_mm', 'pet_mm'. Single station only.
    plot_timescales : timescales (months) to plot as subplots.
    all_timescales : timescales to compute (defaults to 1..12).
    station_dict : optional {station_no: name} mapping.
    start_date, end_date : optional plot range. SPEI is still fitted on the
        full record; only the displayed window is cropped.
    figsize : figure size.
    run_checks : if True, print sanity checks before plotting.

    Returns
    -------
    fig, axes, spei_df : matplotlib figure, axes array, and the full monthly
        SPEI DataFrame (all timescales, not just the plotted ones).
    """
    # Station label
    station_no = df["station"].iloc[0]
    if station_dict and station_no in station_dict:
        station_name = station_dict[station_no]
    else:
        station_name = f"Station {station_no}"

    # Compute monthly aggregation and SPEI for all requested timescales
    monthly = aggregate_to_monthly(df)
    spei_df, _ = calculate_spei_monthly(monthly, timescales=all_timescales)

    if run_checks:
        run_sanity_checks(spei_df, station_name)

    # Crop to plot window after fitting
    spei_plot = spei_df.loc[start_date:end_date] if (start_date or end_date) else spei_df

    # Subplots
    plot_timescales = list(plot_timescales)
    n_plots = len(plot_timescales)
    fig, axes = plt.subplots(n_plots, 1, figsize=figsize, sharex=True)
    if n_plots == 1:
        axes = [axes]

    # Bar width: most of a month, in matplotlib date units (days)
    bar_width = 28

    for idx, k in enumerate(plot_timescales):
        ax = axes[idx]
        col = f"SPEI_{k}"
        if col not in spei_plot.columns:
            ax.text(0.5, 0.5, f"{col} not computed",
                    ha="center", va="center", transform=ax.transAxes)
            continue

        s = spei_plot[col].dropna()
        colors = np.where(s.values >= 0, "#2c7bb6", "#d7191c")  # blue / red

        ax.bar(s.index, s.values, width=bar_width, color=colors,
               align="center", linewidth=0)

        # Reference lines at McKee thresholds
        ax.axhline(0, color="black", linewidth=0.8)
        for y in (-2, -1.5, -1, 1, 1.5, 2):
            ax.axhline(y, color="gray", linestyle="--",
                       linewidth=0.6, alpha=0.6)

        ax.set_ylabel(f"SPEI-{k}", fontsize=14, fontweight="bold")
        ax.grid(True, axis="y", alpha=0.2, linestyle=":", linewidth=0.5)

        # Symmetric y-limits, but capped so a single extreme value doesn't
        # crush the rest of the series.
        ymax = max(3.0, np.nanpercentile(np.abs(s.values), 99) + 0.5)
        ax.set_ylim(-ymax, ymax)

        if idx == 0:
            ax.set_title(station_name, fontsize=20, fontweight="bold", pad=10)

    # X-axis
    axes[-1].set_xlabel("Date", fontsize=14, fontweight="bold")
    axes[-1].xaxis.set_major_locator(YearLocator())
    axes[-1].xaxis.set_major_formatter(DateFormatter("%Y"))
    axes[-1].xaxis.set_minor_locator(MonthLocator(bymonth=(1, 4, 7, 10)))
    axes[-1].tick_params(axis="x", which="minor", length=4,
                         color="gray", width=1)
    axes[-1].tick_params(axis="x", which="major", length=8, width=1.5, labelsize=14)
    plt.setp(axes[-1].get_xticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    return fig, axes, spei_df

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(1, 3, 6, 12),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(1, 3, 6, 12),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(1,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(6,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(1,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    deelen_aligned_df, 
    plot_timescales=(6,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    lelystad_aligned_df, 
    plot_timescales=(1, 3, 6, 12),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    lelystad_aligned_df, 
    plot_timescales=(1,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    lelystad_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    lelystad_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    lelystad_aligned_df, 
    plot_timescales=(6,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    lelystad_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    lelystad_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    de_bilt_aligned_df, 
    plot_timescales=(1, 3, 6, 12),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    de_bilt_aligned_df, 
    plot_timescales=(1,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    de_bilt_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    de_bilt_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    de_bilt_aligned_df, 
    plot_timescales=(6,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    de_bilt_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    de_bilt_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    heino_aligned_df, 
    plot_timescales=(1, 3, 6, 12),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    heino_aligned_df, 
    plot_timescales=(1,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    heino_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    heino_aligned_df, 
    plot_timescales=(3,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    heino_aligned_df, 
    plot_timescales=(6,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    heino_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2010-01-01',
    end_date='2024-12-31',
    figsize=(14,3)
)
plt.show()

In [ ]:
fig, axes, spei_df = plot_spei_monthly(
    heino_aligned_df, 
    plot_timescales=(12,),
    station_dict=no_to_station_dict,
    start_date='2016-01-01',
    end_date='2025-12-31',
    figsize=(14,3)
)
plt.show()

End of the notebook

In [ ]:
# # Old daily z-value 

# # SPEI calculation and plotting
# import matplotlib.pyplot as plt
# import pandas as pd
# import numpy as np
# from scipy import stats
# from matplotlib.dates import MonthLocator, YearLocator, DateFormatter
# from matplotlib.patches import Rectangle

# def calculate_spei_daily(df, timescale_days=30):
#     """
#     Calculate SPEI on daily basis using rolling days with normal distribution over its own
#     timeframe. Direct standardization approach
#     """
#     # Calculate water balance
#     water_balance = df['precip_mm'] - df['pet_mm']
    
#     # Calculate rolling sum for the timescale (in days)
#     accumulated_balance = water_balance.rolling(
#         window=timescale_days, 
#         min_periods=timescale_days
#     ).sum()
    
#     # Calculate mean and std from valid data (non-NaN)
#     valid_data = accumulated_balance.dropna()
#     mean = valid_data.mean()
#     std = valid_data.std()
    
#     # Standardize to SPEI (z-score)
#     spei = (accumulated_balance - mean) / std
    
#     # Convert to Series with proper name
#     spei.name = f'SPEI_{timescale_days}day'
    
#     return spei


# def plot_spei_with_monthly_markers(df, timescales=[1, 3, 6, 12], 
#                                    station_dict=None, 
#                                    start_date=None, 
#                                    end_date=None,
#                                    figsize=(16, 10)):
#     """
#     Plot daily SPEI with monthly markers for multiple timescales
    
#     Parameters:
#     -----------
#     df : DataFrame
#         DataFrame with columns: 'station', 'temp_c', 'precip_mm', 'pet_mm'
#         Index should be datetime
#     timescales : list
#         List of timescales in months (e.g., [1, 3, 6, 12])
#     station_dict : dict, optional
#         Dictionary mapping station numbers to names
#     start_date : str or datetime, optional
#         Start date for plotting (default: all data)
#     end_date : str or datetime, optional
#         End date for plotting (default: all data)
#     figsize : tuple
#         Figure size (width, height)
    
#     Returns:
#     --------
#     fig, axes : matplotlib figure and axes objects
#     """
    
#     # Get station info
#     station_no = df['station'].iloc[0]
#     if station_dict and station_no in station_dict:
#         station_name = station_dict[station_no]
#     else:
#         station_name = f"Station {station_no}"
    
#     # Convert timescales from months to days (approximate)
#     timescales_days = [int(months * 30.44) for months in timescales]  # 30.44 = average days per month
    
#     # Calculate SPEI for each timescale
#     spei_data = {}
#     for months, days in zip(timescales, timescales_days):
#         spei_data[months] = calculate_spei_daily(df, timescale_days=days)
    
#     # Filter date range if specified
#     if start_date or end_date:
#         for months in timescales:
#             spei_data[months] = spei_data[months].loc[start_date:end_date]
    
#     # Create subplots
#     n_plots = len(timescales)
#     fig, axes = plt.subplots(n_plots, 1, figsize=figsize, sharex=True)
    
#     # Handle single plot case
#     if n_plots == 1:
#         axes = [axes]
    
#     # Color scheme
#     colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
    
#     # Plot each timescale
#     for idx, months in enumerate(timescales):
#         ax = axes[idx]
#         spei = spei_data[months]
        
#         # Plot the main SPEI line
#         ax.plot(spei.index, spei.values, 
#                 color=colors[idx % len(colors)], 
#                 linewidth=1.2, 
#                 label=f'SPEI-{months}',
#                 zorder=3)
        
#         # Add monthly markers (vertical lines at month boundaries)
#         # Get month boundaries
#         month_starts = pd.date_range(
#             start=spei.index.min().replace(day=1), 
#             end=spei.index.max(), 
#             freq='MS'  # Month Start
#         )
        
#         for month_start in month_starts:
#             if month_start in spei.index:
#                 # Draw a vertical line at month start
#                 ax.axvline(month_start, color='gray', alpha=0.3, 
#                           linewidth=0.8, linestyle='--', zorder=1)
                
#                 # Add a small marker on the line
#                 spei_value = spei.loc[month_start]
#                 ax.plot(month_start, spei_value, 'o', 
#                        color=colors[idx % len(colors)], 
#                        markersize=4, 
#                        markeredgecolor='white',
#                        markeredgewidth=0.5,
#                        zorder=4)
        
#         # Add horizontal reference lines
#         ax.axhline(y=0, color='black', linestyle='-', alpha=0.5, linewidth=0.8)
#         ax.axhline(y=-1.0, color='orange', linestyle='--', alpha=0.4, linewidth=0.8)
#         ax.axhline(y=-1.5, color='red', linestyle='--', alpha=0.4, linewidth=0.8)
#         ax.axhline(y=-2.0, color='darkred', linestyle='--', alpha=0.4, linewidth=0.8)
#         ax.axhline(y=1.0, color='lightblue', linestyle='--', alpha=0.4, linewidth=0.8)
#         ax.axhline(y=1.5, color='blue', linestyle='--', alpha=0.4, linewidth=0.8)
        
#         # Shade drought/wet zones
#         ax.fill_between(spei.index, -1.0, -1.5, alpha=0.1, color='orange', label='Moderate drought')
#         ax.fill_between(spei.index, -1.5, -2.0, alpha=0.1, color='red', label='Severe drought')
#         ax.fill_between(spei.index, -2.0, spei.min(), alpha=0.1, color='darkred', label='Extreme drought')
#         ax.fill_between(spei.index, 1.0, 1.5, alpha=0.1, color='lightblue', label='Moderate wet')
#         ax.fill_between(spei.index, 1.5, spei.max(), alpha=0.1, color='blue', label='Severe wet')
        
#         # Styling
#         ax.set_ylabel(f'SPEI-{months}', fontsize=11, fontweight='bold')
#         ax.grid(True, alpha=0.3, linestyle=':', linewidth=0.5)
#         ax.set_ylim(spei.min() - 0.5, spei.max() + 0.5)
        
#         # Only show legend on first plot
#         if idx == 0:
#             ax.legend(loc='upper left', fontsize=8, ncol=3)
#             ax.set_title(f'{station_name}', fontsize=14, fontweight='bold', pad=10)
    
#     # Set x-axis label
#     axes[-1].set_xlabel('Date', fontsize=11, fontweight='bold')
    
#     # Format x-axis with major (yearly) and minor (quarterly) ticks
#     axes[-1].xaxis.set_major_locator(YearLocator())
#     axes[-1].xaxis.set_major_formatter(DateFormatter('%Y'))
    
#     # Minor ticks at quarters (Jan, Apr, Jul, Oct)
#     axes[-1].xaxis.set_minor_locator(MonthLocator(bymonth=(1, 4, 7, 10)))
    
#     # Style the ticks
#     axes[-1].tick_params(axis='x', which='minor', length=4, color='gray', width=1)
#     axes[-1].tick_params(axis='x', which='major', length=8, width=1.5)
    
#     plt.xticks(rotation=45, ha='right')
#     plt.tight_layout()
    
#     return fig, axes